# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [3]:
%pip install -qU langchain-core langchain-community langchain-openai langchain-elasticsearch langchain-experimental tiktoken jq lark

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.0 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.3.79 which is incompatible.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>=1.0.0, but you have langchain-text-splitters 0.3.11 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip show langchain-core langchain-community langchain-openai langchain-elasticsearch

Name: langchain-core
Version: 0.3.79
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: jsonpatch, langsmith, packaging, pydantic, PyYAML, tenacity, typing-extensions
Required-by: langchain, langchain-classic, langchain-community, langchain-elasticsearch, langchain-experimental, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt
---
Name: langchain-community
Version: 0.3.31
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tenacity
Required-by: langchain-experimental
---
Name: langchain-openai
Version: 0.3.35
Summary: An integration package connecting OpenAI and LangChain
H

In [2]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [3]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="ironhackworkplace_docs", #give it a meaningful name
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [4]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [10]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def metadata_func(record: dict, metadata: dict) -> dict:
    # Add useful metadata fields for tracing document origin
    metadata["name"] = record.get("name", "unknown")
    metadata["url"] = record.get("url", "N/A")
    metadata["category"] = record.get("category", "general")
    return metadata

In [11]:
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400
)
docs = loader.load_and_split(text_splitter=text_splitter)

In [12]:
for d in docs:
    if "name" not in d.metadata:
        d.metadata["name"] = "unknown"

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [13]:
vectorstore.client.indices.delete(index="ironhackworkplace_docs", ignore_unavailable=True)

documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="ironhackworkplace_docs",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [14]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team at NASA operate?', '3. What are the responsibilities of the sales team at NASA?']


---- Answer ----
The NASA sales team is a part of the Americas region in the sales organization of the company. It is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team is responsible for promoting and selling the company's products and services in the North and South American markets. They work closely with other departments, such as marketing, product development, and customer support, to ensure the company's success in these regions.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

## Iteration 1

In [15]:
ans = chain.invoke("Who leads the NASA sales team, and what regions do they cover?")
print("---- Iteration 1 Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Who is in charge of the NASA sales team and which areas do they oversee?', '2. What are the names of the individuals leading the NASA sales team and which regions are they responsible for?', '3. Can you provide information on the leader of the NASA sales team and the geographical regions they manage?']


---- Iteration 1 Answer ----

The NASA sales team is led by two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America. They cover the United States, Canada, Mexico, Central America, and South America.


In [16]:
ans = chain.invoke("Give me a short summary about NASA's sales team and its key leaders.")
print("---- Iteration 1 (Summary Style) ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ["1. Can you provide a brief overview of NASA's sales team and the individuals who lead it?", '2. What are some key details about the sales team at NASA and the prominent figures who manage it?', "3. I'm interested in learning more about NASA's sales team and the top leaders who drive its success. Can you give me a concise summary?"]


---- Iteration 1 (Summary Style) ----

NASA's sales team is responsible for promoting and selling the company's products and services in the North America South America region. The team is divided into two main regions, with Laura Martinez as the Area Vice-President for North America and Gary Johnson as the Area Vice-President for South America. The team works closely with other departments to ensure high-quality products and services are consistently delivered to customers.


## Iteration 2

In [17]:
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are a helpful assistant that answers clearly and concisely.
Use the context below to answer the question. If the answer isn't found, say "I don't know."

Context:
{context}

Question: "{question}"

Now answer in bullet points:
"""
)

In [18]:
chain = _context | LLM_CONTEXT_PROMPT | llm
ans = chain.invoke("What are the main roles of the NASA sales team?")
print("---- Iteration 2 Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the primary responsibilities of the NASA sales team?', '2. Can you list the key duties of the sales team at NASA?', '3. How would you describe the main functions of the NASA sales team?']


---- Iteration 2 Answer ----

- Promoting and selling company products and services in North America and South America regions
- Establishing relationships with customers and understanding their needs
- Collaborating with other departments to ensure high-quality products and services
- Identifying and pursuing new business opportunities
- Nurturing existing client relationships
- Ensuring customer satisfaction
- Led by two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America


## Iteration 3

In [20]:
ans = chain.invoke("Based on the documents, how does NASA’s sales team structure support its global goals?")
print("---- Iteration 3 ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ["1. How does NASA's sales team structure contribute to achieving its global objectives, as evidenced by the documents?", "2. In what ways does the structure of NASA's sales team align with its global goals, based on the documents?", "3. Can the documents shed light on how NASA's sales team structure is designed to support its global goals?"]


---- Iteration 3 ----

- NASA's sales team is divided into two main regions: North America and South America.
- Each region has its own Area Vice-President, responsible for overseeing sales in their respective areas.
- The sales team collaborates closely with other departments, such as marketing and product development, to ensure high-quality products and services are delivered to clients.
- The team is responsible for identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction.
- The team is also divided into dedicated account managers, sales representatives, and support staff, led by their respective Area Vice-Presidents.
- This structure allows for a focused and targeted approach to sales in different regions, while also promoting collaboration and communication within the team.
